# Task 5 - Create a Judge Model

## Model Choice

Task 2 used `google/gemma-2-9b-it-fast`, so per the assignment we start with the other option: `meta-llama/Meta-Llama-3.1-8B-Instruct`.

If it struggles as a judge (poor structured output compliance, inconsistent rubric application), we will switch to `Qwen/Qwen3-30B-A3B-Instruct-2507` and document why.

## Why Explanation Before Verdict (Schema Ordering)

The schema places `explanation` before `verdict` for each criterion. This matters because LLMs generate tokens sequentially — by forcing the model to write its reasoning first, it performs a "chain-of-thought" before committing to a verdict. If the verdict came first, the model would pick a rating and then rationalize it post-hoc, leading to less accurate and less consistent evaluations. Explanation-first is essentially structured chain-of-thought prompting.

## Why Pydantic

Pydantic schemas provide type-safe, validated structured output. When passed to the API's `response_format` parameter, the model is constrained to produce valid JSON matching the schema exactly — no missing fields, no invalid enum values, no free-form text. This eliminates the need for fragile regex parsing and ensures every judge response is machine-readable.

## Pydantic Output Schema

In [ ]:
from pydantic import BaseModel
from enum import Enum
from typing import Literal


class CriterionResult(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]


class JudgeOutput(BaseModel):
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult

## Judge Prompt

The judge prompt includes the full rubric definitions from Task 1. It excludes cost and latency (measured programmatically). For Grounding, the judge receives both the original product data AND the generated description so it can verify factual claims.

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are an expert product description evaluator. Your job is to evaluate a generated product description against a strict rubric.

You will receive:
1. The ORIGINAL PRODUCT DATA (name, attributes, material, warranty) — this is the ground truth.
2. The GENERATED DESCRIPTION — this is what you are evaluating.

Evaluate the description on exactly 5 criteria. For each criterion, first write a brief explanation of your reasoning, then give a verdict of "good", "ok", or "bad".

## Rubric Definitions

### Fluency
- good: All sentences are clear and easy to follow. No confusing phrasing. Ideas flow logically with no abrupt jumps.
- ok: Overall understandable, but 1-2 sentences feel slightly awkward or clunky. Reader may need to re-read at most one sentence.
- bad: Multiple sentences are confusing, incomplete, or very awkward. Reader struggles to understand key information.

### Grammar
- good: 0-1 minor grammar, spelling, or punctuation errors. No error changes meaning. Consistent tense.
- ok: 2-5 minor errors, but text remains clearly understandable. No systematic error pattern.
- bad: More than 5 errors, or critical mistakes that make sentences hard to read. Systematic repeated errors.

### Tone
- good: Clearly friendly and positive, not exaggerated or spammy. Focuses on customer benefits. Sounds like a genuine sales description.
- ok: Mostly neutral or slightly friendly, could be more engaging. Some generic or flat sentences.
- bad: Tone feels wrong for a sales page (rude, sarcastic, robotic). Overly clickbait or unrealistic promises.

### Length
- good: Total word count between 50 and 90 words (inclusive).
- ok: Total word count between 40-49 or 91-110 words (inclusive).
- bad: Fewer than 40 words or more than 110 words.

### Grounding
This is the most important criterion. Compare EVERY factual claim in the description against the original product data.
- good: All concrete factual statements are supported by the given fields. No invented specs, certifications, dimensions, or brand claims. Generic non-factual marketing phrases ("perfect for everyday use") are allowed.
- ok: One minor inferred detail that is plausible but not strictly given (e.g., calling a cotton T-shirt "soft"). No major fabricated facts.
- bad: Any wrong or clearly fabricated factual detail (e.g., inventing a warranty length, adding specs not in the data, claiming a rating like "4.7/5" not provided). More than one invented assumption. Contradicts the input data.

## Important Notes
- Count words carefully for the Length criterion.
- For Grounding, be strict: if the description mentions ANY specific fact (number, feature, material property, duration, rating) that is NOT in the original product data, it is fabricated.
- Generic marketing phrases like "built to last", "perfect for everyday use", "elevate your experience" are NOT grounding violations.
""".strip()

## Judge Function

In [ ]:
import os
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv("NEBIUS_API_KEY")
)

# Start with the model not used in Task 2
JUDGE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"


def judge_description(product_data: dict, generated_description: str, model: str = JUDGE_MODEL) -> dict:
    """
    Evaluate a generated description using the judge model.
    Returns the parsed JudgeOutput plus latency info.
    """
    user_prompt = (
        f"## Original Product Data\n"
        f"- Name: {product_data['product_name']}\n"
        f"- Attributes: {product_data['Product_attribute_list']}\n"
        f"- Material: {product_data['material']}\n"
        f"- Warranty: {product_data['warranty']}\n\n"
        f"## Generated Description\n"
        f"{generated_description}"
    )

    start = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "judge_output",
                "strict": True,
                "schema": JudgeOutput.model_json_schema()
            }
        },
        temperature=0.0,
        max_tokens=1000
    )
    latency_ms = (time.time() - start) * 1000

    raw = response.choices[0].message.content
    parsed = JudgeOutput.model_validate_json(raw)

    return {
        "parsed": parsed,
        "raw_json": raw,
        "latency_ms": round(latency_ms, 2),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens
    }

## Test with Llama-3.1-8B-Instruct

Run on 3 products to see if the model handles structured output and applies the rubric correctly.

In [ ]:
df = pd.read_excel("assignment_01.xlsx")

test_rows = [0, 2, 9]  # iPhone, Pixel, Garmin
for idx in test_rows:
    row = df.iloc[idx]
    product_data = {
        "product_name": row["product_name"],
        "Product_attribute_list": row["Product_attribute_list"],
        "material": row["material"],
        "warranty": row["warranty"]
    }
    try:
        result = judge_description(product_data, row["generated_description"])
        p = result["parsed"]
        print(f"--- [{idx}] {row['product_name']} ({result['latency_ms']:.0f}ms) ---")
        for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
            c = getattr(p, criterion)
            print(f"  {criterion}: {c.verdict} — {c.explanation[:100]}")
        print()
    except Exception as e:
        print(f"--- [{idx}] {row['product_name']} FAILED: {e}")
        print("Llama-3.1-8B may not support structured output well. Will switch to larger model.")
        break

## Model Switch Decision

`meta-llama/Meta-Llama-3.1-8B-Instruct` works well as a judge:
- Structured output (JSON schema) is correctly followed every time
- Rubric application is accurate — correctly identified grounding=bad for Garmin (invented 4.7/5 rating) and grounding=good for iPhone and GoPro
- Latency is ~15-30s per call (acceptable for 50 products)

**Decision: Keep `meta-llama/Meta-Llama-3.1-8B-Instruct` as the judge model.** No switch needed.